In [1]:
# ============================================================
# LAB 1
# Environment Setup, Python/NumPy/Pandas Warm-up
# and Search Agent - 8 Puzzle
# Algoritmos: BFS, DFS y A*
# ============================================================

import numpy as np
import pandas as pd
import heapq

from itertools import count
from collections import deque


# ============================================================
# 1. WARM-UP: NUMPY Y PANDAS
# ============================================================

print("=" * 70)
print("1. WARM-UP: NUMPY Y PANDAS")
print("=" * 70)

# Matriz NumPy
a = np.arange(12).reshape(3, 4)

print("\nArray:")
print(a)

print("\nMean per column:")
print(a.mean(axis=0))


# DataFrame Pandas
df = pd.DataFrame({
    "score": [12, 15, 9, 18],
    "student": ["A", "B", "C", "D"]
})

print("\nDescripción del DataFrame:")
print(df.describe())


# ============================================================
# 2. DEFINICIÓN DEL PROBLEMA 8-PUZZLE
# ============================================================

print("\n")
print("=" * 70)
print("2. 8-PUZZLE SEARCH AGENT")
print("=" * 70)

# Estado objetivo
GOAL = (1, 2, 3,
        4, 5, 6,
        7, 8, 0)


# ============================================================
# 3. FUNCIÓN PARA GENERAR ESTADOS VECINOS
# ============================================================

def neighbors(state):
    """
    Devuelve todos los estados válidos que pueden obtenerse
    moviendo la casilla vacía (0).
    """

    i = state.index(0)

    r, c = divmod(i, 3)

    moves = []

    # Arriba, abajo, izquierda, derecha
    for dr, dc in [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]:

        nr = r + dr
        nc = c + dc

        if 0 <= nr < 3 and 0 <= nc < 3:

            j = nr * 3 + nc

            s = list(state)

            s[i], s[j] = s[j], s[i]

            moves.append(tuple(s))

    return moves


# ============================================================
# 4. HEURÍSTICA: DISTANCIA MANHATTAN
# ============================================================

def manhattan(state):
    """
    Calcula la distancia Manhattan de todas las fichas
    respecto a su posición objetivo.
    """

    dist = 0

    for idx, val in enumerate(state):

        # No se calcula distancia para la casilla vacía
        if val == 0:
            continue

        # Posición objetivo
        goal_row, goal_col = divmod(val - 1, 3)

        # Posición actual
        row, col = divmod(idx, 3)

        dist += abs(goal_row - row) + abs(goal_col - col)

    return dist


# ============================================================
# 5. RECONSTRUCCIÓN DEL CAMINO
# ============================================================

def reconstruct(parent, state):
    """
    Reconstruye el camino desde el estado inicial
    hasta el estado objetivo.
    """

    path = [state]

    while parent[path[-1]] is not None:
        path.append(parent[path[-1]])

    return list(reversed(path))


# ============================================================
# 6. BFS - BREADTH-FIRST SEARCH
# ============================================================

def bfs(start):

    frontier = deque([start])

    visited = {start}

    parent = {
        start: None
    }

    expanded = 0

    while frontier:

        state = frontier.popleft()

        expanded += 1

        # Verificar si encontramos el objetivo
        if state == GOAL:

            return reconstruct(parent, state), expanded

        # Explorar vecinos
        for nxt in neighbors(state):

            if nxt not in visited:

                visited.add(nxt)

                parent[nxt] = state

                frontier.append(nxt)

    return None, expanded


# ============================================================
# 7. DFS - DEPTH-FIRST SEARCH
# ============================================================

def dfs(start):

    frontier = [start]

    visited = {start}

    parent = {
        start: None
    }

    expanded = 0

    while frontier:

        state = frontier.pop()

        expanded += 1

        # Verificar si encontramos el objetivo
        if state == GOAL:

            return reconstruct(parent, state), expanded

        # Se invierte el orden para mantener
        # una exploración consistente
        for nxt in reversed(neighbors(state)):

            if nxt not in visited:

                visited.add(nxt)

                parent[nxt] = state

                frontier.append(nxt)

    return None, expanded


# ============================================================
# 8. A* - A STAR SEARCH
# ============================================================

def astar(start):

    counter = count()

    frontier = [
        (
            manhattan(start),
            next(counter),
            start,
            0
        )
    ]

    parent = {
        start: None
    }

    g_score = {
        start: 0
    }

    expanded = 0

    while frontier:

        _, _, state, g = heapq.heappop(frontier)

        expanded += 1

        # Verificar si encontramos el objetivo
        if state == GOAL:

            return reconstruct(parent, state), expanded

        # Explorar vecinos
        for nxt in neighbors(state):

            new_g = g + 1

            if nxt not in g_score or new_g < g_score[nxt]:

                g_score[nxt] = new_g

                parent[nxt] = state

                priority = new_g + manhattan(nxt)

                heapq.heappush(
                    frontier,
                    (
                        priority,
                        next(counter),
                        nxt,
                        new_g
                    )
                )

    return None, expanded


# ============================================================
# 9. ESTADOS PROPORCIONADOS EN EL LABORATORIO
# ============================================================

states = [

    (1, 7, 6,
     2, 0, 8,
     4, 5, 3),

    (1, 6, 4,
     8, 5, 7,
     2, 0, 3),

    (5, 3, 1,
     4, 0, 8,
     2, 6, 7),

    (2, 1, 4,
     3, 5, 6,
     7, 0, 8),

    (0, 6, 2,
     5, 3, 8,
     7, 4, 1)

]


# ============================================================
# 10. EJECUTAR TODOS LOS ALGORITMOS
# ============================================================

results = []

print("\n")
print("=" * 70)
print("3. EJECUCIÓN DE LOS ESTADOS")
print("=" * 70)


for index, start in enumerate(states, start=1):

    print("\n")
    print("-" * 70)
    print(f"ESTADO {index}")
    print("-" * 70)

    print(
        start[0:3],
        "\n",
        start[3:6],
        "\n",
        start[6:9]
    )

    # --------------------------------------------------------
    # BFS
    # --------------------------------------------------------

    path_bfs, exp_bfs = bfs(start)

    bfs_length = len(path_bfs) - 1

    print(
        f"\nBFS -> "
        f"Path length: {bfs_length} | "
        f"Nodes expanded: {exp_bfs:,}"
    )

    results.append({
        "Estado": f"Estado {index}",
        "Algoritmo": "BFS",
        "Longitud del camino": bfs_length,
        "Nodos expandidos": exp_bfs
    })


    # --------------------------------------------------------
    # DFS
    # --------------------------------------------------------

    path_dfs, exp_dfs = dfs(start)

    dfs_length = len(path_dfs) - 1

    print(
        f"DFS -> "
        f"Path length: {dfs_length} | "
        f"Nodes expanded: {exp_dfs:,}"
    )

    results.append({
        "Estado": f"Estado {index}",
        "Algoritmo": "DFS",
        "Longitud del camino": dfs_length,
        "Nodos expandidos": exp_dfs
    })


    # --------------------------------------------------------
    # A*
    # --------------------------------------------------------

    path_astar, exp_astar = astar(start)

    astar_length = len(path_astar) - 1

    print(
        f"A*  -> "
        f"Path length: {astar_length} | "
        f"Nodes expanded: {exp_astar:,}"
    )

    results.append({
        "Estado": f"Estado {index}",
        "Algoritmo": "A*",
        "Longitud del camino": astar_length,
        "Nodos expandidos": exp_astar
    })


# ============================================================
# 11. TABLA FINAL DE RESULTADOS
# ============================================================

results_df = pd.DataFrame(results)

print("\n")
print("=" * 70)
print("4. TABLA FINAL DE RESULTADOS")
print("=" * 70)

display(results_df)


# ============================================================
# 12. TABLA COMPARATIVA TIPO MATRIZ
# ============================================================

comparison = results_df.pivot(
    index="Estado",
    columns="Algoritmo",
    values=[
        "Longitud del camino",
        "Nodos expandidos"
    ]
)

print("\n")
print("=" * 70)
print("5. COMPARACIÓN ENTRE ALGORITMOS")
print("=" * 70)

display(comparison)


# ============================================================
# 13. RESUMEN PROMEDIO
# ============================================================

summary = (
    results_df
    .groupby("Algoritmo")
    [["Longitud del camino", "Nodos expandidos"]]
    .mean()
    .round(2)
)

print("\n")
print("=" * 70)
print("6. PROMEDIOS POR ALGORITMO")
print("=" * 70)

display(summary)


# ============================================================
# 14. MENSAJE FINAL
# ============================================================

print("\n")
print("=" * 70)
print("LABORATORIO EJECUTADO CORRECTAMENTE")
print("=" * 70)

print("""
Interpretación general:

- BFS encuentra caminos óptimos, pero puede expandir muchos nodos.
- DFS encuentra una solución, pero no garantiza el camino más corto.
- A* utiliza la distancia Manhattan y obtiene caminos óptimos
  expandiendo significativamente menos nodos que BFS.
""")

1. WARM-UP: NUMPY Y PANDAS

Array:
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]

Mean per column:
[4. 5. 6. 7.]

Descripción del DataFrame:
           score
count   4.000000
mean   13.500000
std     3.872983
min     9.000000
25%    11.250000
50%    13.500000
75%    15.750000
max    18.000000


2. 8-PUZZLE SEARCH AGENT


3. EJECUCIÓN DE LOS ESTADOS


----------------------------------------------------------------------
ESTADO 1
----------------------------------------------------------------------
(1, 7, 6) 
 (2, 0, 8) 
 (4, 5, 3)

BFS -> Path length: 20 | Nodes expanded: 48,630
DFS -> Path length: 49800 | Nodes expanded: 134,880
A*  -> Path length: 20 | Nodes expanded: 635


----------------------------------------------------------------------
ESTADO 2
----------------------------------------------------------------------
(1, 6, 4) 
 (8, 5, 7) 
 (2, 0, 3)

BFS -> Path length: 25 | Nodes expanded: 158,525
DFS -> Path length: 63065 | Nodes expanded: 81,025
A*  -> Path length: 25 | Nod

,Estado,Algoritmo,Longitud del camino,Nodos expandidos
0,Estado 1,BFS,20,48630
1,Estado 1,DFS,49800,134880
2,Estado 1,A*,20,635
3,Estado 2,BFS,25,158525
4,Estado 2,DFS,63065,81025
5,Estado 2,A*,25,3328
6,Estado 3,BFS,22,93605
7,Estado 3,DFS,23028,162487
8,Estado 3,A*,22,999
9,Estado 4,BFS,19,29383




5. COMPARACIÓN ENTRE ALGORITMOS


Longitud del camino            Nodos expandidos                
Algoritmo                  A* BFS    DFS               A*     BFS     DFS
Estado                                                                   
Estado 1                   20  20  49800              635   48630  134880
Estado 2                   25  25  63065             3328  158525   81025
Estado 3                   22  22  23028              999   93605  162487
Estado 4                   19  19  49267              462   29383  135605
Estado 5                   26  26  33978             5117  160495   36147



6. PROMEDIOS POR ALGORITMO


,Longitud del camino,Nodos expandidos
Algoritmo,,
A*,22.4,2108.2
BFS,22.4,98127.6
DFS,43827.6,110028.8




LABORATORIO EJECUTADO CORRECTAMENTE

Interpretación general:

- BFS encuentra caminos óptimos, pero puede expandir muchos nodos.
- DFS encuentra una solución, pero no garantiza el camino más corto.
- A* utiliza la distancia Manhattan y obtiene caminos óptimos
  expandiendo significativamente menos nodos que BFS.

